<a href="https://colab.research.google.com/github/hasinduwelikala/northstar-databases-analytics/blob/main/01_SQL_in_R_NorthStar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
install.packages("sqldf")
install.packages("ggplot2")

library(sqldf)
library(ggplot2)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [8]:
base_url <- "https://raw.githubusercontent.com/hasinduwelikala/northstar-databases-analytics/main/data/"

files <- c("orders.csv", "deliveries.csv", "hubs.csv",
           "customers.csv", "complaints.csv", "drivers.csv",
           "vehicles.csv", "incidents.csv", "app_events.csv",
           "data_dictionary.csv")

for (f in files) {
  download.file(paste0(base_url, f), destfile = f, quiet = TRUE)
}
cat("All files downloaded successfully.\n")

orders     <- read.csv("orders.csv",     stringsAsFactors = FALSE, na.strings = "")
deliveries <- read.csv("deliveries.csv", stringsAsFactors = FALSE, na.strings = "")
hubs       <- read.csv("hubs.csv",       stringsAsFactors = FALSE, na.strings = "")
customers  <- read.csv("customers.csv",  stringsAsFactors = FALSE, na.strings = "")
complaints <- read.csv("complaints.csv", stringsAsFactors = FALSE, na.strings = "")
drivers    <- read.csv("drivers.csv", stringsAsFactors = FALSE, na.strings = "")
vehicles   <- read.csv("vehicles.csv", stringsAsFactors = FALSE, na.strings = "")
incidents  <- read.csv("incidents.csv", stringsAsFactors = FALSE, na.strings = "")
app_events <- read.csv("app_events.csv", stringsAsFactors = FALSE, na.strings = "")
data_dictionary <- read.csv("data_dictionary.csv", stringsAsFactors = FALSE, na.strings = "")

All files downloaded successfully.


In [10]:
standardise_zone <- function(x) {
  x <- trimws(x)
  x <- gsub("(?i)^ctr$",       "Central",   x, perl = TRUE)
  x <- gsub("(?i)^riverside$", "Riverside", x, perl = TRUE)
  x <- gsub("(?i)^airport$",   "Airport",   x, perl = TRUE)
  x <- gsub("(?i)^central$",   "Central",   x, perl = TRUE)
  x <- gsub("(?i)^north$",     "North",     x, perl = TRUE)
  x <- gsub("(?i)^south$",     "South",     x, perl = TRUE)
  x <- gsub("(?i)^east$",      "East",      x, perl = TRUE)
  x <- gsub("(?i)^west$",      "West",      x, perl = TRUE)
  return(x)
}

orders$pickup_zone  <- standardise_zone(orders$pickup_zone)
orders$dropoff_zone <- standardise_zone(orders$dropoff_zone)
customers$home_zone <- standardise_zone(customers$home_zone)
hubs$zone           <- standardise_zone(hubs$zone)

In [11]:
first_rows <- sqldf("SELECT * FROM orders LIMIT 3")
print(first_rows)

# Order deliveries by fuel cost highest first
ordered_deliveries <- sqldf("
  SELECT *
  FROM deliveries
  ORDER BY fuel_or_charge_cost DESC
  LIMIT 10
")
print(ordered_deliveries)

  order_id customer_id service_type    order_created_at promised_window_hours
1   O00001       C0292    Passenger 2024-08-20 14:43:00                     6
2   O00002       C0459    Passenger 2024-05-14 22:16:00                    24
3   O00003       C0161    Passenger 2025-09-02 14:37:00                     4
  pickup_zone dropoff_zone priority_level order_value booking_channel
1     Airport        South         Medium      126.65             App
2       North      Airport            Low      109.30             App
3        West      Airport           High       33.50           Phone
  special_handling_flag
1                     0
2                     0
3                     0
   delivery_id order_id driver_id vehicle_id hub_id       dispatch_time
1      DL00897   O00672      D084       V043    H07 2025-01-04 00:01:00
2      DL00144   O00170      D016       V016    H06 2024-06-24 09:02:00
3      DL00713   O00728      D044       V039    H03 2024-10-21 06:15:00
4      DL00664   O00430 

In [15]:
on_time_by_hub <- sqldf("
  SELECT
    h.zone                                                    AS hub_zone,
    h.hub_name,
    COUNT(d.delivery_id)                                      AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'OnTime'  THEN 1 ELSE 0 END) AS on_time_count,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_count,
    SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END) AS failed_count,
    ROUND(
      100.0 * SUM(CASE WHEN d.delivery_status = 'OnTime' THEN 1 ELSE 0 END)
      / COUNT(d.delivery_id), 1
    ) AS on_time_pct
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  JOIN hubs   h ON d.hub_id   = h.hub_id
  GROUP BY h.zone, h.hub_name
  ORDER BY on_time_pct ASC
")
print(on_time_by_hub)

   hub_zone       hub_name total_deliveries on_time_count delayed_count
1   Central   Central Core              115            67            25
2   Airport    Airport Hub              104            62            27
3   Central  Midtown Relay              128            80            22
4      West      West Gate              127            83            28
5     South     South Link              106            70            26
6 Riverside  Riverside Hub              115            76            25
7     North North Exchange              136            93            26
8      East      East Dock              119            85            23
  failed_count on_time_pct
1           23        58.3
2           15        59.6
3           26        62.5
4           16        65.4
5           10        66.0
6           14        66.1
7           17        68.4
8           11        71.4


In [16]:
avg_cost_by_service <- sqldf("
  SELECT
    o.service_type,
    COUNT(d.delivery_id)                             AS total_deliveries,
    ROUND(AVG(d.fuel_or_charge_cost), 2)             AS avg_cost,
    ROUND(MIN(d.fuel_or_charge_cost), 2)             AS min_cost,
    ROUND(MAX(d.fuel_or_charge_cost), 2)             AS max_cost,
    ROUND(AVG(d.route_distance_km), 2)               AS avg_distance_km,
    ROUND(AVG(d.fuel_or_charge_cost /
          NULLIF(d.route_distance_km, 0)), 4)        AS cost_per_km
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  WHERE d.fuel_or_charge_cost IS NOT NULL
  GROUP BY o.service_type
  ORDER BY avg_cost DESC
")
print(avg_cost_by_service)

  service_type total_deliveries avg_cost min_cost max_cost avg_distance_km
1     Business              126    13.14     2.78    27.38           13.63
2       Parcel              230    13.08     2.61    26.99           14.35
3       Retail              224    12.97     2.50    29.43           14.31
4      Medical              108    12.77     2.84    24.54           13.23
5    Passenger              262    12.40     2.50    23.07           13.60
  cost_per_km
1      1.2351
2      1.2934
3      1.2308
4      1.4083
5      1.2125


In [17]:
top_complainers <- sqldf("
  SELECT
    c.customer_id,
    cu.customer_type,
    cu.home_zone,
    cu.loyalty_score,
    COUNT(c.complaint_id)                                  AS complaint_count,
    SUM(CASE WHEN c.severity = 'High' THEN 1 ELSE 0 END)  AS high_severity_count,
    ROUND(AVG(c.resolution_days), 1)                       AS avg_resolution_days,
    ROUND(SUM(COALESCE(c.compensation_amount, 0)), 2)      AS total_compensation
  FROM complaints c
  JOIN customers cu ON c.customer_id = cu.customer_id
  WHERE cu.home_zone IS NOT NULL
  GROUP BY c.customer_id, cu.customer_type, cu.home_zone, cu.loyalty_score
  HAVING COUNT(c.complaint_id) >= 2
  ORDER BY complaint_count DESC, high_severity_count DESC
  LIMIT 15
")
print(top_complainers)

   customer_id customer_type home_zone loyalty_score complaint_count
1        C0368      Consumer     North          49.5               4
2        C0372      Consumer      West          26.2               3
3        C0421      Consumer   Central          59.0               3
4        C0573           SME   Airport          57.3               3
5        C0110      Consumer      East            NA               3
6        C0142      Consumer     South          47.0               3
7        C0172      Consumer     North          75.4               3
8        C0242      Consumer      East          83.8               3
9        C0626      Consumer     South          61.6               3
10       C0191      Consumer     North          58.9               3
11       C0282      Consumer Riverside          71.4               3
12       C0545      Consumer     South          66.9               3
13       C0015           SME     South          54.5               2
14       C0078      Consumer      

In [18]:
failure_by_priority <- sqldf("
  SELECT
    o.priority_level,
    o.service_type,
    COUNT(d.delivery_id)                                          AS total,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
    ROUND(
      100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END)
      / COUNT(d.delivery_id), 1
    ) AS failure_rate_pct
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.priority_level, o.service_type
  ORDER BY
    CASE o.priority_level
      WHEN 'Critical' THEN 1
      WHEN 'High'     THEN 2
      WHEN 'Medium'   THEN 3
      WHEN 'Low'      THEN 4
    END,
    failure_rate_pct DESC
")
print(failure_by_priority)


   priority_level service_type total failed failure_rate_pct
1        Critical      Medical    10      1             10.0
2        Critical     Business    10      1             10.0
3        Critical    Passenger    19      1              5.3
4        Critical       Retail    18      0              0.0
5        Critical       Parcel    17      0              0.0
6            High      Medical    21      4             19.0
7            High     Business    35      6             17.1
8            High       Retail    52      8             15.4
9            High    Passenger    61      9             14.8
10           High       Parcel    62      5              8.1
11         Medium     Business    55     15             27.3
12         Medium      Medical    40      8             20.0
13         Medium       Parcel    89     14             15.7
14         Medium       Retail    97     13             13.4
15         Medium    Passenger   105     13             12.4
16            Low    Pas

In [19]:
distinct_failed_zones <- sqldf("
  SELECT DISTINCT o.dropoff_zone
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  WHERE d.delivery_status = 'Failed'
  ORDER BY o.dropoff_zone
")
print(distinct_failed_zones)

  dropoff_zone
1      Airport
2      Central
3         East
4        North
5    Riverside
6        South
7         West


In [20]:
above_avg_cost <- sqldf("
  SELECT
    d.delivery_id,
    o.service_type,
    o.pickup_zone,
    o.dropoff_zone,
    d.fuel_or_charge_cost,
    d.delivery_status
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  WHERE d.fuel_or_charge_cost > (
    SELECT AVG(fuel_or_charge_cost)
    FROM deliveries
    WHERE fuel_or_charge_cost IS NOT NULL
  )
  ORDER BY d.fuel_or_charge_cost DESC
  LIMIT 20
")
print(above_avg_cost)

   delivery_id service_type pickup_zone dropoff_zone fuel_or_charge_cost
1      DL00897       Retail     Airport    Riverside               29.43
2      DL00144     Business     Airport      Airport               27.38
3      DL00713       Parcel       South    Riverside               26.99
4      DL00664       Retail     Central        South               25.46
5      DL00119     Business     Airport        South               25.09
6      DL00052      Medical     Airport    Riverside               24.54
7      DL00287       Parcel     Airport        North               24.50
8      DL00806      Medical     Airport      Airport               24.27
9      DL00090       Retail     Airport        North               24.20
10     DL00373       Parcel       South         West               23.60
11     DL00818       Retail     Airport         West               23.18
12     DL00721    Passenger     Airport         West               23.07
13     DL00477       Retail     Airport      Centra

In [21]:
worst_rated <- sqldf("
  SELECT
    d.delivery_id,
    o.service_type,
    h.hub_name,
    d.delivery_status,
    d.customer_rating_post_delivery,
    d.manual_route_override_count
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  JOIN hubs   h ON d.hub_id   = h.hub_id
  WHERE d.customer_rating_post_delivery IS NOT NULL
  ORDER BY d.customer_rating_post_delivery ASC
  LIMIT 10 OFFSET 0
")
print(worst_rated)

   delivery_id service_type       hub_name delivery_status
1      DL00017      Medical   Central Core         Delayed
2      DL00259     Business    Airport Hub         Delayed
3      DL00288      Medical      East Dock         Delayed
4      DL00479       Retail North Exchange         Delayed
5      DL00509    Passenger North Exchange         Delayed
6      DL00558       Retail   Central Core          Failed
7      DL00195       Parcel    Airport Hub         Delayed
8      DL00536     Business North Exchange          Failed
9      DL00392       Parcel      West Gate         Delayed
10     DL00057       Retail      West Gate          Failed
   customer_rating_post_delivery manual_route_override_count
1                           1.00                           0
2                           1.00                           1
3                           1.00                           0
4                           1.00                           0
5                           1.00              

In [22]:
hub_scorecard <- sqldf("
  SELECT
    h.hub_name,
    h.zone,
    h.capacity_score,
    COUNT(d.delivery_id)                                              AS total_deliveries,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'OnTime'  THEN 1 ELSE 0 END)
          / COUNT(d.delivery_id), 1)                                  AS on_time_pct,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END)
          / COUNT(d.delivery_id), 1)                                  AS failure_pct,
    ROUND(AVG(d.fuel_or_charge_cost), 2)                              AS avg_cost,
    ROUND(AVG(d.fuel_or_charge_cost / NULLIF(d.route_distance_km, 0)), 4) AS cost_per_km,
    ROUND(AVG(d.manual_route_override_count), 3)                      AS avg_overrides,
    SUM(d.proof_of_completion_missing)                                AS missing_proofs,
    ROUND(AVG(
      CASE WHEN d.customer_rating_post_delivery IS NOT NULL
           THEN d.customer_rating_post_delivery END
    ), 2)                                                             AS avg_rating
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  JOIN hubs   h ON d.hub_id   = h.hub_id
  GROUP BY h.hub_name, h.zone, h.capacity_score
  ORDER BY failure_pct DESC, avg_overrides DESC
")
print(hub_scorecard)

        hub_name      zone capacity_score total_deliveries on_time_pct
1  Midtown Relay   Central             63              128        62.5
2   Central Core   Central             88              115        58.3
3    Airport Hub   Airport             71              104        59.6
4      West Gate      West             69              127        65.4
5 North Exchange     North             82              136        68.4
6  Riverside Hub Riverside             66              115        66.1
7     South Link     South             78              106        66.0
8      East Dock      East             74              119        71.4
  failure_pct avg_cost cost_per_km avg_overrides missing_proofs avg_rating
1        20.3    11.71      1.1818         1.109             10       3.88
2        20.0    13.69      1.4776         0.948             10       3.67
3        14.4    13.32      1.2392         0.913             10       3.88
4        12.6    13.17      1.3554         0.874             